# OVCA leave-one-cell-type-out: `cellina` vs a labels baseline (OOD + ID), leakage-safe split

A port of [`03_loo_eval.ipynb`](03_loo_eval.ipynb) from the CRC (`crc_232`) CosMx dataset to the
**OVCA** Xenium Prime ovarian-cancer slide, built to host a third model conditioned on **per-cell
H&E embeddings** (section D, pending — see below).

The experiment keeps 03's structure: hold out one cell type in one domain, never showing that
combination during training, then ask each model to predict expression for control cells placed
counterfactually into that domain, and score `cellina` against the no-context
`wsi.LabelsBaseline` floor — split into **OOD** (the held-out type) and **ID** (everything else).

| | CRC (`03_loo_eval`) | OVCA (this notebook) |
|---|---|---|
| control -> target | `REF` -> `CRC` | `Smooth muscle` -> `Tumor` |
| held-out cell type | `Myeloid` | `Stromal` |
| cells | 111,989 | 14,170 |
| genes | 2,000 HVGs | 5,101-gene Xenium panel, **no HVG subsetting** (5,090 after QC) |
| split | by cell | **by cell _and_ by ROI** (see below) |

`Smooth muscle -> Tumor` holding out `Stromal` is the OVCA analogue of CRC's myeloid-in-cancer
setup: quiescent stroma reprogramming into cancer-associated fibroblasts.

## The split is ROI-level, not just cell-level

CRC's `split_indices` holds out cells. That is safe for models that see only counts and labels,
but it leaks badly for anything conditioned on per-cell H&E tiles: tiles are ~112um wide while
cells sit 7-9um apart, so **neighbouring cells' tiles overlap by >90%**. Holding out a Stromal
cell while training on the Malignant cell beside it hands the model that Stromal cell's pixels
anyway. `scripts/extract_cell_embeddings_roi.py` placed the 36 circular ROIs far enough apart to
make an ROI-level split pixel-disjoint, and its docstring is explicit that downstream splits must
go *by `roi_id`, never by cell*.

So `eval.split_indices_roi_holdout` withholds **both**:

- **every cell in Tumor ROIs `[9, 6, 3, 8]`** — closes the pixel leak, and
- **every `(Stromal, Tumor)` cell slide-wide** — preserves the leave-one-cell-type-out property.

Scoring then happens only on the Tumor-domain cells *inside* those withheld ROIs
(`obs["is_scored_target"]`), so every scored cell is both expression-unseen and pixel-unseen:

- **OOD** — the 560 held-out `Stromal` cells: `(Stromal, Tumor)` was never seen, in any ROI.
- **ID** — the 1,684 held-in types: their `(cell type, Tumor)` combination *was* seen, but these
  particular cells' tiles were not. So ID here measures generalisation to unseen tissue, which is
  stricter than a normal LOO's ID regime.

Those four ROIs are the Tumor ROIs with the most Stromal cells (181/134/129/116 = 560, versus 13
for the worst pair). That is a **statistical-power** choice, made on cell abundance and never on
any model's score — so it cannot bias one model against another — but it does narrow the OOD
claim to stroma-rich tumour regions.

The `labels` baseline has no pixels and so gains nothing from the ROI holdout; it pays for it
anyway, because all models must share one split for the table to mean anything.

## Sections

| Section | Produces |
|---|---|
| **A. Shared setup + cellina** | `observed.h5ad`, `cellina_pred.h5ad` |
| **B. Labels baseline** | `labels_pred.h5ad` |
| **C. Compare (OOD + ID)** | metric tables + plot |
| **D. H&E-embedding model** | *pending — needs `src/wsi/embedding_baseline.py`* |

Section D will condition on `[onehot(cell_type), uni2]` — the same decoder as `labels`, with the
domain one-hot swapped for the cell's own H&E morphology. Because it has no encoder it is a
**direct predictor, not a counterfactual**: it reads the very cells it predicts, so it is a
*ceiling* on what H&E carries, not a competitor. Expect it to beat `cellina`; that says nothing
about `cellina`. The `kind` column in section C exists to keep that distinction attached to the
numbers.

All the LOO plumbing (preprocessing, splits, the metrics glue) is imported from `../scripts` —
the notebook only orchestrates. Everything runs in **one kernel** (`cellina_edge`): A, then B,
then C, top to bottom.

In [ ]:
import os
import sys

import numpy as np

# ../scripts for the LOO plumbing (eval/metrics); ../src for the `wsi` package
# (LabelsBaseline), which is not pip-installed into cellina_edge.
sys.path.append(os.path.abspath("../scripts"))
sys.path.append(os.path.abspath("../src"))

from utils import set_seed
from eval import (
    load_ovca_loo_bundle,
    preprocess_ovca,
    preprocess_spatial_features,
    split_indices_roi_holdout,  #key for leakage prevention
    make_comparison_adata,
    run_loo_eval,
    COUNTS_PER_K,
)
from configs.adata_ovca_config import ADATA_ARGS

set_seed(0)

# --- experiment config (see configs/adata_ovca_config.py) ------------------
# Bundle built by scripts/prepare_ovca_loo_data.py under lazyslide-env -- see its header for
# why the notebook can't read OVCA_with_embeddings.zarr directly (zarr v3 / Arrow strings).
BUNDLE_DIR = os.path.abspath("../data/ovca_loo")

LABELS_KEY = ADATA_ARGS["labels_key"]              # "cell_type"
DOMAINS_KEY = ADATA_ARGS["domains_key"]            # "domain"        ground truth; drives arms
PRED_DOMAINS_KEY = ADATA_ARGS["pred_domains_key"]  # "pred_domain"   carried along, unused here

HOLDOUT_CT = ADATA_ARGS["holdout_celltype"]        # "Stromal"
CONTROL_LABEL = ADATA_ARGS["control_domains"][0]   # "Smooth muscle" (quiescent stroma)
HOLDOUT_LABEL = ADATA_ARGS["holdout_domains"][0]   # "Tumor"         (held out for Stromal)
HOLDOUT_ROIS = ADATA_ARGS["holdout_rois"]          # [9, 6, 3, 8] -- whole Tumor ROIs withheld

PRED_DIR = "preds_ovca_embed"      # small cache for the cross-env prediction handoff
os.makedirs(PRED_DIR, exist_ok=True)

## A. Shared setup + cellina  *(`cellina_edge` kernel)*

In [ ]:
# Load the prepared bundle (raw counts + micron centroids + true/predicted domain + the uni2 /
# virchow2 per-cell H&E embeddings), then the shared QC. Unlike preprocess_crc there is no label
# remap or coord assembly to do -- the bundle arrives clean. n_top_genes=None: the Xenium panel
# is already curated, so no HVG subsetting.
adata = load_ovca_loo_bundle(BUNDLE_DIR)
adata = preprocess_ovca(
    adata,
    n_top_genes=ADATA_ARGS["n_top_genes"],
    labels_key=LABELS_KEY,
    domains_key=DOMAINS_KEY,
    pred_domains_key=PRED_DOMAINS_KEY,
)
adata

# Expect 14170 x 5090 (filter_genes drops 11 of the 5101; no cells are dropped),
# with obsm: spatial, uni2 (14170, 1536), virchow2 (14170, 2560)

In [ ]:
# Stromal cells in tumor domain remain unseen in training, and several ROIs later used in testing are held out to avoid leakakge 
train_idx, val_idx, test_idx = split_indices_roi_holdout(adata, HOLDOUT_CT, HOLDOUT_ROIS, 
    labels_key=LABELS_KEY, domains_key=DOMAINS_KEY, 
    holdout_domains=ADATA_ARGS["holdout_domains"], 
    seed=0
)

print(f"train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

#Expect train=10408 val=1156 test=2606 

train=96259  val=10695  test=5035


In [ ]:
# Spatial neighbour graph + aggregated features
adata = preprocess_spatial_features(adata, n_neighbors=ADATA_ARGS["n_neighbors"], test_indices=test_idx, bandwidth=ADATA_ARGS["bandwidth_um"])

In [ ]:
# Control = ALL Smooth-muscle cells (they stay in training -- they are the counterfactual's
# INPUT, not its target, so there is nothing to protect). Observed target = the Tumor-domain
# cells INSIDE the withheld ROIs (obs["is_scored_target"], set by split_indices_roi_holdout).
# Restricting the target to those ROIs is what keeps the pixel-leak fix intact at scoring time.
# One pass over every cell type, then split scoring (section C) into:
#   - OOD: the held-out cell type (Stromal) -> (Stromal, Tumor) was never seen in training
#   - ID : the held-in cell types           -> (cell type, Tumor) was seen, but these cells'
#          H&E tiles were not (their whole ROI was withheld)
is_holdout_ct = adata.obs[LABELS_KEY].astype(str) == HOLDOUT_CT
mask_control = (adata.obs[DOMAINS_KEY].astype(str) == CONTROL_LABEL).values
mask_target  = adata.obs["is_scored_target"].values
idx_control = np.where(mask_control)[0]
idx_target  = np.where(mask_target)[0]

# Cell-type labels per arm, so we can slice OOD vs ID after scoring. The counterfactual moves
# control cells into Tumor but keeps each cell's own type, so cellina's / labels' predicted arm
# carries the CONTROL cells' types.
ct_control = adata.obs[LABELS_KEY].astype(str).values[mask_control]
ct_target  = adata.obs[LABELS_KEY].astype(str).values[mask_target]

# Donor pool for the Tumor microenvironment: observed neighbours of the target cells, excluding
# the held-out cell type (Stromal) so the held-out cells never leak in as donors. These donors
# sit in the withheld ROIs, so cellina never trained on their expression -- it only reads them
# at inference, which is exactly the edge-perturbation design.
conn = adata.obsp["spatial_connectivities_orig"]
neighbor_indices = np.unique(conn[idx_target].nonzero()[1])
neighbor_indices = neighbor_indices[~is_holdout_ct.values[neighbor_indices]]
print(f"control({CONTROL_LABEL})={idx_control.size}  "
      f"target({HOLDOUT_LABEL}, withheld ROIs)={idx_target.size}  "
      f"donor-neighbours={neighbor_indices.size}  OOD-celltype={HOLDOUT_CT!r}")

# Expect control=2803  target=2244  (of which OOD Stromal=560, ID other types=1684)

In [ ]:
# Observed pair (ground truth): real control vs real target counts across all cell types, with
# the cell-type label attached so section C can slice OOD (held-out type) from ID (held-in).
observed = make_comparison_adata(
    adata.layers["counts"][mask_control, :],
    adata.layers["counts"][mask_target, :],
    adata.var_names, control_label=CONTROL_LABEL, perturbed_label=HOLDOUT_LABEL,
    pert_col=DOMAINS_KEY
)
observed.obs[LABELS_KEY] = np.concatenate([ct_control, ct_target])
observed.write(os.path.join(PRED_DIR, "observed.h5ad"))
observed

# Expect 5047 x 5090  (2803 control + 2244 target)

In [ ]:
# --- Train cellina (tutorial-style; args from configs/cellina_config.py) ---
from cellina import Cellina
from configs.cellina_config import (
    MODEL_ARGS as CELLINA_MODEL_ARGS,
    TRAIN_ARGS as CELLINA_TRAIN_ARGS,
    PLAN_KWARGS as CELLINA_PLAN_KWARGS,
)

Cellina.setup_anndata(
    adata,  batch_key=ADATA_ARGS["batch_key"], labels_key=LABELS_KEY, domains_key=DOMAINS_KEY,
    spatial_obsm_key="spatial_x", layer="counts",
)
cellina_model = Cellina(adata, **CELLINA_MODEL_ARGS)

train_args = dict(CELLINA_TRAIN_ARGS)
train_args["datasplitter_kwargs"] = {"external_indexing": [train_idx, val_idx, test_idx]}
cellina_model.train(**train_args, plan_kwargs=CELLINA_PLAN_KWARGS)

INFO     Generating sequential column names                                                                        
INFO     cellina: The Cellina model has been initialized with adversarial domain forgetting                        


/data/ddimitrov/software/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/data/ddimitrov/software/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_data

Training:   0%|          | 0/100 [00:00<?, ?it/s]

Monitored metric vae_loss_validation did not improve in the last 10 records. Best score: 324.833. Signaling Trainer to stop.


In [ ]:
# Edge-perturbation counterfactual over ALL control cells: rewire each into the (withheld-ROI)
# Tumor microenvironment (one pass; sliced into OOD/ID at scoring time).
cellina_cf = cellina_model.get_counterfactual_expression(
    indices=idx_control, batch_size=2048, seed=0,
    neighbour_indices=neighbor_indices, precomputed=False,
)
# cellina's own reconstruction of the control cells (lets us also use a predicted baseline).
cellina_ctrl = cellina_model.get_normalized_expression(
    adata[mask_control], library_size=COUNTS_PER_K, batch_size=2048,
)

cellina_pred = make_comparison_adata(
    cellina_ctrl, cellina_cf, adata.var_names,
    control_label=CONTROL_LABEL, perturbed_label=HOLDOUT_LABEL, pert_col=DOMAINS_KEY,
)
# Both arms are the control cells (reconstruction + their counterfactual).
cellina_pred.obs[LABELS_KEY] = np.concatenate([ct_control, ct_control])
cellina_pred.write(os.path.join(PRED_DIR, "cellina_pred.h5ad"))
cellina_pred

## B. Labels baseline  *(same `cellina_edge` kernel)*

The labels baseline reuses the same in-memory `adata` and splits from section A. We register it
with its own `setup_anndata` (cell type + domain as categoricals, raw counts), train the
label-conditioned decoder, then run the counterfactual over **all `Smooth muscle` cells →
`Tumor`** (each cell keeps its own cell-type label) — the same one-pass inference cellina got, so
the OOD/ID split in section C is apples-to-apples.

Its counterfactual is a label swap rather than a spatial rewire: take the control cells, relabel
their domain to `Tumor`, decode. It sees no expression context and no spatial context — the
no-context floor `cellina` should beat, especially OOD.

In [ ]:
# Train the labels baseline on the same split (reuses adata / splits from section A).
from wsi.labels_baseline import LabelsBaseline
from configs.labels_config import (
    MODEL_ARGS as LABELS_MODEL_ARGS,
    TRAIN_ARGS as LABELS_TRAIN_ARGS,
    PLAN_KWARGS as LABELS_PLAN_KWARGS,
)

LabelsBaseline.setup_anndata(
    adata, labels_key=LABELS_KEY, domains_key=DOMAINS_KEY,
    batch_key=ADATA_ARGS["batch_key"], layer="counts",
)
labels_model = LabelsBaseline(adata, **LABELS_MODEL_ARGS)

train_args = dict(LABELS_TRAIN_ARGS)
train_args["datasplitter_kwargs"] = {"external_indexing": [train_idx, val_idx, test_idx]}
labels_model.train(**train_args, plan_kwargs=LABELS_PLAN_KWARGS)

In [ ]:
# Counterfactual over ALL control cells -> Tumor (one pass), plus control reconstruction.
labels_cf = labels_model.get_counterfactual_expression(
    idx_control, target_domain=HOLDOUT_LABEL, batch_size=2048,
)  # observed (latent) library size; each cell keeps its own cell-type label
labels_ctrl = labels_model.get_normalized_expression(
    adata[mask_control], library_size=COUNTS_PER_K, batch_size=2048,
)

labels_pred = make_comparison_adata(
    labels_ctrl, labels_cf, adata.var_names,
    control_label=CONTROL_LABEL, perturbed_label=HOLDOUT_LABEL, pert_col=DOMAINS_KEY,
)
labels_pred.obs[LABELS_KEY] = np.concatenate([ct_control, ct_control])
labels_pred.write(os.path.join(PRED_DIR, "labels_pred.h5ad"))
labels_pred

## C. Compare — OOD + ID

Load the cached predictions and score each model with `compare_adatas` on the top-50
observed-effect genes, **split by regime**:

- **OOD** — the 560 held-out `Stromal` cells in the withheld Tumor ROIs: predicting
  `(Stromal, Tumor)`, a combination never seen in training, on tissue never seen either.
- **ID** — the 1,684 held-in types in those same ROIs: their `(cell type, Tumor)` combinations
  were seen in training, but these cells' H&E tiles were not.

Both regimes reuse the *same* one-pass predictions from sections A/B; we simply slice the
observed/predicted AnnDatas by the attached `cell_type`. `run_loo_eval` uses
`predicted_logfc_baseline="observed_control"`, so — as in the cellina tutorial — both the
ground-truth and predicted logFC are measured against the *same observed* control population
(within each regime's cell set). Note this means the **predicted control arm is not read** by the
table at all; only the scatter below uses it (`predicted_control`).

The `kind` column records how each model reaches its prediction — `counterfactual` (never sees
the target cells) vs `direct` (reads the cells it predicts, i.e. a ceiling). With only cellina
and labels present every row is `counterfactual`; it matters once section D lands.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc

sys.path.append(os.path.abspath("../scripts"))
from eval import run_loo_eval
from metrics import get_perturbation_effects

# Re-declared so section C runs standalone in a fresh kernel.
PRED_DIR = "preds_ovca_embed"
DOMAINS_KEY, LABELS_KEY = "domain", "cell_type"
CONTROL_LABEL, HOLDOUT_LABEL, HOLDOUT_CT = "Smooth muscle", "Tumor", "Stromal"

observed     = sc.read(os.path.join(PRED_DIR, "observed.h5ad"))
cellina_pred = sc.read(os.path.join(PRED_DIR, "cellina_pred.h5ad"))
labels_pred  = sc.read(os.path.join(PRED_DIR, "labels_pred.h5ad"))

PREDS = [("cellina", cellina_pred), ("labels", labels_pred)]

# How each model gets to its prediction. Both current models are counterfactual: they never see
# the target cells. A direct H&E-embedding model (section D) would be "direct" -- it reads the
# very cells it predicts, so it is a CEILING, not a peer. Keep this column honest as models are
# added; it is what stops the table being misread once it is copied out of the notebook.
REGIME = {"cellina": "counterfactual", "labels": "counterfactual"}

def score(mask_fn):
    """Score each model's control -> Tumor prediction on the cells selected by mask_fn."""
    res = []
    for name, pred in PREDS:
        o = observed[mask_fn(observed)].copy()
        p = pred[mask_fn(pred)].copy()
        row = run_loo_eval(o, p, pert_col=DOMAINS_KEY, control_label=CONTROL_LABEL).iloc[0]
        res.append(row.rename(name))
    return pd.DataFrame(res)

# OOD = held-out cell type (Stromal, never seen in Tumor); ID = held-in types (seen in Tumor,
# but scored only on cells whose ROI -- and therefore whose H&E tiles -- were withheld).
comparison_ood = score(lambda a: (a.obs[LABELS_KEY].astype(str) == HOLDOUT_CT).values)
comparison_id  = score(lambda a: (a.obs[LABELS_KEY].astype(str) != HOLDOUT_CT).values)

comparison = pd.concat({"OOD": comparison_ood, "ID": comparison_id}, names=["regime", "model"])
comparison.insert(0, "kind", [REGIME[m] for m in comparison.index.get_level_values("model")])
comparison

In [ ]:
# Observed vs predicted logFC scatter (cellina) on the OOD held-out Stromal, top DE genes.
# NOTE this is the only place the PREDICTED control arm is read: it passes
# predicted_logfc_baseline="predicted_control", whereas run_loo_eval above defaults to
# "observed_control" and ignores it.
import matplotlib.pyplot as plt
import numpy as np

ood = lambda a: a[(a.obs[LABELS_KEY].astype(str) == HOLDOUT_CT).values].copy()
obs_eff, pred_eff = get_perturbation_effects(
    ood(observed), ood(cellina_pred), pert_col=DOMAINS_KEY, control_label=CONTROL_LABEL,
    mode="logfc", predicted_logfc_baseline="predicted_control",
)
o = obs_eff.loc[HOLDOUT_LABEL].values
p = pred_eff.loc[HOLDOUT_LABEL].values
deg = np.argsort(-np.abs(o))[:50]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(o, p, s=8, color="lightgray", alpha=0.4)
ax.scatter(o[deg], p[deg], s=30, color="#d62728")
lims = [min(o.min(), p.min()), max(o.max(), p.max())]
ax.plot(lims, lims, "k--", lw=1, alpha=0.6)
ax.set_xlabel(f"Observed logFC ({CONTROL_LABEL} -> {HOLDOUT_LABEL})")
ax.set_ylabel("Predicted logFC (cellina counterfactual)")
ax.set_title(f"cellina: observed vs predicted logFC (OOD {HOLDOUT_CT})")
plt.tight_layout()
plt.show()